In [158]:
# ===============================
# 1. Import libraries
# ===============================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import cross_val_score, TimeSeriesSplit, KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack

from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('wordnet', quiet=True)

import warnings
warnings.filterwarnings('ignore')

SEED = 42

print("All libraries loaded successfully ✅")

All libraries loaded successfully ✅


In [159]:



# Project root (notebook is inside notebooks/)
PROJECT_ROOT = Path("..").resolve()

# Input files
DATA_DIR = PROJECT_ROOT / "data"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

# Output files
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

SUBMISSION_PATH_050 = OUTPUT_DIR / "submission_threshold_050.csv"
SUBMISSION_PATH_072 = OUTPUT_DIR / "submission_threshold_072.csv"

# Load train data
df = pd.read_csv(TRAIN_PATH)

print("Train file:", TRAIN_PATH)
print("Test file:", TEST_PATH)
print("Output dir:", OUTPUT_DIR)

Train file: /Users/atulbharti/Downloads/Hertie_School_study_Docs/Semester_2/ML/homework-3-ab/data/train.csv
Test file: /Users/atulbharti/Downloads/Hertie_School_study_Docs/Semester_2/ML/homework-3-ab/data/test.csv
Output dir: /Users/atulbharti/Downloads/Hertie_School_study_Docs/Semester_2/ML/homework-3-ab/outputs


In [160]:
#loading the train.csv dataset

df = pd.read_csv("../data/train.csv")

df.head()

,Id,livestream_id,username,text,date,time,channelname,subscriber_bi,IS_TOXIC
0,12,1,admirallove,"""""jan 6th was a peaceful protest""""",2023-08-02,2023-08-02 18:47:46+00:00,sondsol,1.0,0
1,13,1,i8urmom2,"unprecedented? it was a """"mostly peaceful"""" riot.",2023-08-02,2023-08-02 18:47:53+00:00,sondsol,0.0,0
2,14,1,i8urmom2,"unprecedented riots are those left wing """"prot...",2023-08-02,2023-08-02 18:48:15+00:00,sondsol,0.0,0
3,15,1,i8urmom2,"@AdmiralLove it was, by far. far far far far f...",2023-08-02,2023-08-02 18:48:44+00:00,sondsol,0.0,0
4,16,1,internet_gas,The prosecutor was between 2015 to 2016 Zloche...,2023-08-02,2023-08-02 18:48:45+00:00,sondsol,0.0,0


In [161]:
#Parsing date and time
df["time"] = pd.to_datetime(df["time"], errors="coerce")


df = df.sort_values("time").reset_index(drop=True)

print(df["time"].iloc[0], "to", df["time"].iloc[-1])

df.head()



2023-08-02 18:45:24+00:00 to 2023-09-11 23:36:07+00:00


,Id,livestream_id,username,text,date,time,channelname,subscriber_bi,IS_TOXIC
0,439,2,koshkathemerc,Ahhhhhh looks like I'm not collecting my bet t...,2023-08-02,2023-08-02 18:45:24+00:00,tyt,1.0,0
1,443,2,eurylino,She's the most disappointing Kraken of all tim...,2023-08-02,2023-08-02 18:45:24+00:00,tyt,1.0,0
2,440,2,hoosierdaddy89,Sickening Six,2023-08-02,2023-08-02 18:45:24+00:00,tyt,0.0,0
3,441,2,hoosierdaddy89,Like the Chicago 7,2023-08-02,2023-08-02 18:45:24+00:00,tyt,0.0,0
4,442,2,hoosierdaddy89,need a name,2023-08-02,2023-08-02 18:45:24+00:00,tyt,0.0,0


## Train/Validation Split
Splitting the data right now **before any feature engineering**, using 95/05 split. All feature engineering will then be fitted on the training set only, and applied to the validation and test set to avoid any form of **data leakage** where information from the test set influences how we build features. Because the data is ordered by time, we take the first 95% as training, the next 5% as validation. 


We split the data using `GroupShuffleSplit` based on `livestream_id`. This ensures that all comments from the same livestream session are assigned entirely to either the training set or the validation set, reducing leakage across splits.

This is important because comments within the same livestream can share common context, style, and discussion topics. A grouped split therefore gives a more realistic estimate of generalisation to unseen livestream sessions.

In [162]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(test_size=0.05, random_state=42)

train_indices, val_indices = next(
    splitter.split(df, groups=df["livestream_id"])
)

train_data = df.iloc[train_indices].copy()
validation_data = df.iloc[val_indices].copy()

y_train = train_data["IS_TOXIC"].values
y_validation = validation_data["IS_TOXIC"].values

print(f"Training samples:   {len(train_data):,}")
print(f"Validation samples: {len(validation_data):,}")
print()
print(f"Train date range: {train_data['time'].iloc[0].date()} to {train_data['time'].iloc[-1].date()}")
print(f"Val date range:   {validation_data['time'].iloc[0].date()} to {validation_data['time'].iloc[-1].date()}")

Training samples:   115,125
Validation samples: 4,692

Train date range: 2023-08-02 to 2023-09-11
Val date range:   2023-08-02 to 2023-09-06


## Part 1: Regex Feature Engineering

*Feature Engineering* is the process of creating new features from the existing ones. This can be applied to any data, but is particularly useful when data is not numerical. Text, image data, and audio data are all examples of data that can be feature engineered.

In our case, since we are working on the twitch comments, will have to convert raw text into feature that can be used to predict the toxicity. I will use '**Regex**

### A quick regex refresher

| Pattern | Matches |
|---|---|
| `[A-Z]` | any uppercase letter |
| `\w+` | one or more word characters |
| `\s+` | one or more whitespace characters |
| `https?://\S+` | a URL starting with http or https |
| `@\w+` | a Twitter @mention |
| `[!?]` | exclamation mark or question mark |

Below I have defined a function that when given a column of text will return a bunch of new features based on Regex patterns. I have added a few to start, see if you can add some of the ones we discussed.

Note: In python `re.findall(pattern, text)` returns a list of all matches so `len(re.findall(...))` gives us a count of matches.


In [163]:
def extract_regex_features(text_series):
    """
    Extract hand-crafted features from a pandas Series of tweet strings.
    Returns a DataFrame with one row per tweet and one column per feature.
    """
    features = pd.DataFrame()
    features['char_count'] = text_series.str.len() # This calulates the length of each twitch comment
    features['word_count'] = text_series.str.split().str.len() # This calculates the number of words in each twitch comment
    
    features['caps_ratio'] = text_series.apply(
        lambda t: len(re.findall(r'[A-Z]', t)) / max(len(re.findall(r'[a-zA-Z]', t)), 1)
    )  # This calculates the ratio of uppercase characters to total characters
    
    features['exclamation_count'] = text_series.apply(lambda t: len(re.findall(r'!', t))) # This calculates the number of exclamation marks
    features['question_count']    = text_series.apply(lambda t: len(re.findall(r'\?', t))) # This calculates the number of question marks
    features['has_url']           = text_series.apply(
        lambda t: int(bool(re.search(r'https?://\S+|www\.\S+', t))) # This checks if the tweet contains a URL
    )
    
    features['mention_count']  = text_series.apply(lambda t: len(re.findall(r'@\w+', t))) # This calculates the number of mentions
    features['hashtag_count']  = text_series.apply(lambda t: len(re.findall(r'#\w+', t))) # This calculates the number of hashtags
    features['repeated_chars'] = text_series.apply(
        lambda t: int(bool(re.search(r'(.)\1{2,}', t))) # This checks if the tweet contains a character repeated more than twice
    )
    return features

# Apply to all four splits
X_regex_train = extract_regex_features(train_data['text'])
X_regex_val   = extract_regex_features(validation_data['text'])

X_regex_train.head(20)

,char_count,word_count,caps_ratio,exclamation_count,question_count,has_url,mention_count,hashtag_count,repeated_chars
0,52,9,0.046512,0,0,0,0,0,1
1,108,19,0.046512,0,0,0,0,0,0
2,13,2,0.166667,0,0,0,0,0,0
3,18,4,0.142857,0,0,0,0,0,0
4,11,3,0.000000,0,0,0,0,0,0
5,149,25,0.016529,1,0,0,0,0,0
6,12,2,0.222222,0,0,0,0,0,0
7,63,9,0.020408,3,0,0,1,0,1
8,26,4,0.100000,0,0,0,0,0,0
9,67,17,0.060000,0,0,0,0,0,0


In [164]:
#check if any of these features differ meaningfully between toxic and non-toxic comments in teh training set 
regex_with_target = X_regex_train.copy()
regex_with_target["IS_TOXIC"] = y_train

# Group by toxicity and take mean
regex_with_target.groupby("IS_TOXIC").mean()

,char_count,word_count,caps_ratio,exclamation_count,question_count,has_url,mention_count,hashtag_count,repeated_chars
IS_TOXIC,,,,,,,,,
0,40.448165,6.910524,0.186234,0.081158,0.108572,0.023597,0.110827,0.001497,0.063979
1,54.639627,10.113738,0.128715,0.095065,0.093246,0.001940,0.108767,0.000364,0.064993


## Part 2: Datetime Feature Engineering

Chat behavior on livestream platforms like Twitch can vary depending on when comments are posted. For example, users may behave differently late at night compared to during the day, or on weekends compared to weekdays.

However, the raw `time` column is not directly useful for machine learning, since each timestamp is unique and cannot be interpreted numerically by the model.

To make this information usable, we extract meaningful components from the timestamp.

### Why I am doing this

We extract datetime features for three main reasons:

1. **Capture behavioral patterns over time**  
   Toxicity may vary by hour of day or day of week.

2. **Reduce complexity of raw timestamps**  
   Instead of using full timestamps, we extract interpretable features like hour and weekday.

3. **Provide additional signals beyond text**  
   These features complement text-based features (TF-IDF) by adding contextual information.

The features we extract are:
- **hour**: hour of the day (0–23)
- **dayofweek**: day of the week (0 = Monday, 6 = Sunday)
- **is_weekend**: whether the comment was posted on a weekend

In [165]:
# First ensure time is datetime
train_data["time"] = pd.to_datetime(train_data["time"], errors="coerce")
validation_data["time"] = pd.to_datetime(validation_data["time"], errors="coerce")

# Extract features directly into dataset
for df_split in [train_data, validation_data]:
    df_split["hour"] = df_split["time"].dt.hour
    df_split["day_of_week"] = df_split["time"].dt.dayofweek
    df_split["is_weekend"] = df_split["day_of_week"].isin([5, 6]).astype(int)

In [166]:
#check if it worked
train_data[["time", "hour", "day_of_week", "is_weekend"]].head()

,time,hour,day_of_week,is_weekend
0,2023-08-02 18:45:24+00:00,18,2,0
1,2023-08-02 18:45:24+00:00,18,2,0
2,2023-08-02 18:45:24+00:00,18,2,0
3,2023-08-02 18:45:24+00:00,18,2,0
4,2023-08-02 18:45:24+00:00,18,2,0


### Sine/cosine (cyclical) encoding

The solution is to encode cyclical variables using sine and cosine. For a variable with period $T$:

$$\text{sin\_enc} = \sin\left(\frac{2\pi \cdot x}{T}\right), \quad \text{cos\_enc} = \cos\left(\frac{2\pi \cdot x}{T}\right)$$

Together, these two values uniquely identify any point in the cycle, and points that are close in time are also close in the encoded space. Note that you always need **both** sin and cos as neither alone is sufficient to uniquely identify a time (e.g. sin gives the same value for hour 2 and hour 10). Use the equations above to fill in the blanks in the function below. Then apply the encoding to the data, storing the result in `train_data`, and `validation_data`.

In [167]:
def add_cyclical_features(df_in):
    out = pd.DataFrame(index=df_in.index)
    
    out['hour_sin'] = np.sin(2 * np.pi * df_in['hour'] / 24)
    out['hour_cos'] = np.cos(2 * np.pi * df_in['hour'] / 24)
    out['dow_sin']  = np.sin(2 * np.pi * df_in['day_of_week'] / 7)
    out['dow_cos']  = np.cos(2 * np.pi * df_in['day_of_week'] / 7)
    out["is_weekend"] = df_in["is_weekend"]
    
    return out

X_dt_train = add_cyclical_features(train_data)
X_dt_val   = add_cyclical_features(validation_data)

X_dt_train.head()

,hour_sin,hour_cos,dow_sin,dow_cos,is_weekend
0,-1.0,-1.836970e-16,0.974928,-0.222521,0
1,-1.0,-1.836970e-16,0.974928,-0.222521,0
2,-1.0,-1.836970e-16,0.974928,-0.222521,0
3,-1.0,-1.836970e-16,0.974928,-0.222521,0
4,-1.0,-1.836970e-16,0.974928,-0.222521,0


## Part 3: Text Preprocessing and TF-IDF

Our regex-based features gave us a useful starting point, but they only capture a limited part of the information in the text. In a text classification problem, the actual words and short phrases used in a comment are often much more informative.

To make use of this, we apply a standard NLP technique called **TF-IDF**. TF-IDF converts text into numerical features by representing each comment as a weighted combination of words and phrases. This allows the model to learn which terms are more associated with toxic versus non-toxic comments.

Before vectorising the text, we apply some light preprocessing. Twitch comments may contain URLs, mentions, repeated characters, punctuation, and other noisy elements. Some cleaning helps reduce unnecessary variation in the text while keeping the main signal intact.

### Why we are doing this

We do this for three reasons:

1. **Machine learning models cannot use raw text directly**  
   The text column must first be converted into numbers.

2. **We want to capture the meaning and content of comments**  
   Unlike regex features, TF-IDF uses the full vocabulary of the corpus.

3. **We want to reduce noisy variation in the text**  
   Simple normalization helps the model treat similar comments more consistently.

In this notebook, we use a light preprocessing strategy so that we do not remove too much Twitch-specific signal.

In [168]:


# make sure nltk wordnet is installed (run once)
# import nltk
# nltk.download('wordnet')
# nltk.download('omw-1.4')

STOPWORDS = {
    'a','about','above','after','again','all','am','an','and','any','are','as','at',
    'be','because','been','before','being','below','between','both','by','can','did',
    'do','does','doing','down','during','each','few','for','from','further','had',
    'has','have','having','he','her','here','hers','herself','him','himself','his',
    'how','i','if','in','into','is','it','its','itself','just','me','more','most',
    'my','myself','now','of','on','once','only','or','other','our','ours','ourselves',
    'out','own','same','she','should','so','some','such','than','that','the','their',
    'theirs','them','themselves','then','there','these','they','this','those','through',
    'to','too','under','until','up','very','was','we','were','what','when','where',
    'which','while','who','whom','why','will','with','you','your','yours','yourself',
    'yourselves'
}

def preprocess(text_series):
    lemmatizer  = WordNetLemmatizer()
    
    url_pat     = r'https?://\S+|www\.\S+'
    mention_pat = r'@\w+'
    alpha_pat   = r'[^a-zA-Z\s]'
    repeat_pat  = r'(.)\1{2,}'
    
    cleaned = []
    
    for text in text_series:
        text = text.lower()
        text = re.sub(url_pat,     ' URL ',  text)
        text = re.sub(mention_pat, ' USER ', text)
        text = re.sub(alpha_pat,   ' ',      text)
        text = re.sub(repeat_pat,  r'\1\1',  text)
        
        words = [
            lemmatizer.lemmatize(w)
            for w in text.split()
            if len(w) > 1 and w not in STOPWORDS
        ]
        
        cleaned.append(' '.join(words))
    
    return cleaned

In [169]:
print("Preprocessing text...")

train_clean = preprocess(train_data["text"])
val_clean   = preprocess(validation_data["text"])

print("Done.")

print("\nBefore:", train_data["text"].iloc[0])
print("After: ", train_clean[0])

Preprocessing text...
Done.

Before: Ahhhhhh looks like I'm not collecting my bet tonight
After:  ahh look like not collecting bet tonight


### TF-IDF Vectorisation

**TF-IDF** stands for Term Frequency–Inverse Document Frequency. It converts a collection of text documents into a matrix where each row is a document and each column is a word (or n-gram), with values that reflect how important that word is to that document relative to the whole corpus.

The intuition: a word that appears frequently in one tweet but rarely across the whole dataset is more informative than one that appears everywhere. TF-IDF captures this by upweighting rare words and downweighting common ones.

In [170]:
vectoriser = TfidfVectorizer(ngram_range=(1, 3), max_features=150000, min_df=2, max_df=0.95, sublinear_tf=True)

X_tfidf_train = vectoriser.fit_transform(train_clean)
X_tfidf_val   = vectoriser.transform(val_clean)


print(f'Vocabulary size (from training data): {len(vectoriser.vocabulary_):,}')
print(f'TF-IDF train shape: {X_tfidf_train.shape}')
print(f'TF-IDF val shape:   {X_tfidf_val.shape}')
print()
print("Sample features from vocabulary:")
print(vectoriser.get_feature_names_out()[:20])

Vocabulary size (from training data): 61,297
TF-IDF train shape: (115125, 61297)
TF-IDF val shape:   (4692, 61297)

Sample features from vocabulary:
['aa' 'aah' 'aand' 'aaron' 'aaww' 'ab' 'abandon' 'abandon hope'
 'abandoned' 'abandoning' 'abba' 'abbot' 'abbott' 'abby' 'abc' 'abducted'
 'abe' 'abe lincoln' 'abhorrent' 'abide']


## Part 4: Fitting a Model

Now we have three feature sets, all correctly fitted on training data and applied to all three splits:

| Feature set | Train | Val |
|---|---|---|
| Regex | `X_regex_train` | `X_regex_val` |
| Datetime | `X_dt_train` | `X_dt_val` |
| TF-IDF | `X_tfidf_train` | `X_tfidf_val` |

### Combining feature sets

To use all three together we need to horizontally stack them into a single feature matrix. The regex and datetime features are dense NumPy arrays; the TF-IDF matrix is sparse. We use `scipy.sparse.hstack` to combine them efficiently without converting the sparse matrix to dense (which would use a lot of memory).

We also scale the Regex features as not all of them are binary.

In [171]:
# Rebuild scaler on 14 cols only (no user features)
scaler = StandardScaler()

X_dense_train = scaler.fit_transform(
    np.hstack([X_regex_train.values, X_dt_train.values])
)
X_dense_val = scaler.transform(
    np.hstack([X_regex_val.values, X_dt_val.values])
)

X_train_all = hstack([X_tfidf_train, X_dense_train])
X_val_all   = hstack([X_tfidf_val,   X_dense_val])

print(f"Train shape: {X_train_all.shape}")  # (115125, 61311)
print(f"Val shape:   {X_val_all.shape}")    # (4692,   61311)

Train shape: (115125, 61311)
Val shape:   (4692, 61311)


In [172]:
# ── User toxicity rate — built on training data only ──────────────────
for col in ["user_toxic_rate", "user_comment_count"]:
    if col in train_data.columns:
        train_data = train_data.drop(columns=[col])
    if col in validation_data.columns:
        validation_data = validation_data.drop(columns=[col])

user_stats = train_data.groupby("username")["IS_TOXIC"].agg(
    user_toxic_rate="mean",
    user_comment_count="count"
).reset_index()

global_toxic_rate = y_train.mean()

train_data      = train_data.merge(user_stats, on="username", how="left")
validation_data = validation_data.merge(user_stats, on="username", how="left")

for df_split in [train_data, validation_data]:
    df_split["user_toxic_rate"]    = df_split["user_toxic_rate"].fillna(global_toxic_rate)
    df_split["user_comment_count"] = df_split["user_comment_count"].fillna(0)

print(f"Global toxic rate: {global_toxic_rate:.4f}")
print(train_data[["username","user_toxic_rate","user_comment_count"]].head())

Global toxic rate: 0.0716
         username  user_toxic_rate  user_comment_count
0   koshkathemerc         0.014085                  71
1        eurylino         0.032258                  31
2  hoosierdaddy89         0.176471                  17
3  hoosierdaddy89         0.176471                  17
4  hoosierdaddy89         0.176471                  17


In [173]:
# XGBoost works on dense features only — no TF-IDF
user_cols = ["user_toxic_rate", "user_comment_count", "subscriber_bi"]

X_xgb_train = np.hstack([
    X_regex_train.values,           # 9 cols
    X_dt_train.values,              # 5 cols
    train_data[user_cols].values    # 3 cols
])                                  # total = 17 cols

X_xgb_val = np.hstack([
    X_regex_val.values,
    X_dt_val.values,
    validation_data[user_cols].values
])

scaler_xgb = StandardScaler()
X_xgb_train = scaler_xgb.fit_transform(X_xgb_train)
X_xgb_val   = scaler_xgb.transform(X_xgb_val)

feature_names = list(X_regex_train.columns) + list(X_dt_train.columns) + user_cols
print(f"XGBoost train shape: {X_xgb_train.shape}")
print(f"XGBoost val shape:   {X_xgb_val.shape}")
print(f"Features: {feature_names}")

XGBoost train shape: (115125, 17)
XGBoost val shape:   (4692, 17)
Features: ['char_count', 'word_count', 'caps_ratio', 'exclamation_count', 'question_count', 'has_url', 'mention_count', 'hashtag_count', 'repeated_chars', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'user_toxic_rate', 'user_comment_count', 'subscriber_bi']


In [174]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

neg = sum(y_train == 0)
pos = sum(y_train == 1)
scale = neg / pos
print(f"scale_pos_weight: {scale:.2f}")

xgb_model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=scale,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss',
    early_stopping_rounds=50,
    verbosity=0
)

xgb_model.fit(
    X_xgb_train, y_train,
    eval_set=[(X_xgb_val, y_validation)],
    verbose=False
)

print(f"Best iteration: {xgb_model.best_iteration}")
print("Training complete ✅")

scale_pos_weight: 12.96
Best iteration: 998
Training complete ✅


In [175]:
lr_all = LogisticRegression(
    max_iter=1000,
    random_state=SEED,
    class_weight='balanced'
)
lr_all.fit(X_train_all, y_train)

y_proba_lr = lr_all.predict_proba(X_val_all)[:, 1]

best_acc, best_t = 0, 0.5
for t in np.arange(0.1, 0.95, 0.01):
    acc = accuracy_score(y_validation, (y_proba_lr >= t).astype(int))
    if acc > best_acc:
        best_acc = acc
        best_t   = t

print(f"Best threshold: {best_t:.2f}  |  Accuracy: {best_acc:.4f}")

Best threshold: 0.66  |  Accuracy: 0.9768


In [176]:
# Get probabilities from both models
y_proba_lr  = lr_all.predict_proba(X_val_all)[:, 1]     # your existing LR
y_proba_xgb = xgb_model.predict_proba(X_xgb_val)[:, 1]  # new XGBoost

# Try different ensemble weights
print("=" * 55)
print(f"{'Weight LR':<12} {'Weight XGB':<12} {'Threshold':<12} {'Accuracy'}")
print("=" * 55)

results = []
for lr_w in [0.6, 0.7, 0.8, 0.9, 1.0]:
    xgb_w = round(1.0 - lr_w, 1)
    y_proba_ens = lr_w * y_proba_lr + xgb_w * y_proba_xgb

    best_acc, best_t = 0, 0.5
    for t in np.arange(0.1, 0.95, 0.01):
        acc = accuracy_score(y_validation, (y_proba_ens >= t).astype(int))
        if acc > best_acc:
            best_acc = acc
            best_t   = t

    results.append((lr_w, xgb_w, best_t, best_acc))
    print(f"{lr_w:<12} {xgb_w:<12} {best_t:<12.2f} {best_acc:.4f}")

# Pick best combination
best = max(results, key=lambda x: x[3])
print(f"\nBest: LR={best[0]}, XGB={best[1]}, threshold={best[2]:.2f}, accuracy={best[3]:.4f}")

Weight LR    Weight XGB   Threshold    Accuracy
0.6          0.4          0.59         0.9774
0.7          0.3          0.63         0.9798
0.8          0.2          0.64         0.9798
0.9          0.1          0.62         0.9780
1.0          0.0          0.66         0.9768

Best: LR=0.7, XGB=0.3, threshold=0.63, accuracy=0.9798


In [177]:
#Final evaluation for best ensemble
best_lr_w, best_xgb_w, best_t, _ = best

y_proba_final = best_lr_w * y_proba_lr + best_xgb_w * y_proba_xgb
y_pred_final  = (y_proba_final >= best_t).astype(int)

print(f"Ensemble ({best_lr_w} LR + {best_xgb_w} XGB) @ threshold {best_t:.2f}")
print(f"Accuracy: {accuracy_score(y_validation, y_pred_final):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_validation, y_proba_final):.4f}")
print()
print(classification_report(y_validation, y_pred_final,
      target_names=["Negative", "Toxic"]))
print(confusion_matrix(y_validation, y_pred_final))

Ensemble (0.7 LR + 0.3 XGB) @ threshold 0.63
Accuracy: 0.9798
ROC-AUC:  0.9760

              precision    recall  f1-score   support

    Negative       0.99      0.99      0.99      4335
       Toxic       0.89      0.84      0.86       357

    accuracy                           0.98      4692
   macro avg       0.94      0.91      0.93      4692
weighted avg       0.98      0.98      0.98      4692

[[4298   37]
 [  58  299]]


In [178]:
#changing the threshold to 0.72
# Try threshold 0.72 with best ensemble weights
y_pred_072 = (y_proba_final >= 0.72).astype(int)

print(f"Ensemble ({best_lr_w} LR + {best_xgb_w} XGB) @ threshold 0.72")
print(f"Accuracy: {accuracy_score(y_validation, y_pred_072):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_validation, y_proba_final):.4f}")
print()
print(classification_report(y_validation, y_pred_072,
      target_names=["Negative", "Toxic"]))
print(confusion_matrix(y_validation, y_pred_072))

Ensemble (0.7 LR + 0.3 XGB) @ threshold 0.72
Accuracy: 0.9738
ROC-AUC:  0.9760

              precision    recall  f1-score   support

    Negative       0.98      1.00      0.99      4335
       Toxic       0.93      0.71      0.80       357

    accuracy                           0.97      4692
   macro avg       0.95      0.85      0.89      4692
weighted avg       0.97      0.97      0.97      4692

[[4317   18]
 [ 105  252]]


In [179]:
print(f"{'Threshold':<12} {'Accuracy':<12} {'Toxic Precision':<18} {'Toxic Recall':<15} {'Toxic F1'}")
print("=" * 70)

for t in np.arange(0.50, 0.90, 0.01):
    y_pred_t = (y_proba_final >= t).astype(int)
    acc = accuracy_score(y_validation, y_pred_t)
    report = classification_report(y_validation, y_pred_t, 
                                   output_dict=True, 
                                   target_names=["Negative", "Toxic"])
    prec  = report["Toxic"]["precision"]
    rec   = report["Toxic"]["recall"]
    f1    = report["Toxic"]["f1-score"]
    print(f"{t:<12.2f} {acc:<12.4f} {prec:<18.4f} {rec:<15.4f} {f1:.4f}")

Threshold    Accuracy     Toxic Precision    Toxic Recall    Toxic F1
0.50         0.9691       0.7477             0.8964          0.8153
0.51         0.9706       0.7601             0.8964          0.8226
0.52         0.9721       0.7743             0.8936          0.8296
0.53         0.9742       0.7935             0.8936          0.8406
0.54         0.9753       0.8035             0.8936          0.8462
0.55         0.9766       0.8175             0.8908          0.8525
0.56         0.9785       0.8386             0.8880          0.8626
0.57         0.9778       0.8391             0.8768          0.8575
0.58         0.9787       0.8521             0.8711          0.8615
0.59         0.9793       0.8631             0.8655          0.8643
0.60         0.9791       0.8669             0.8571          0.8620
0.61         0.9795       0.8761             0.8515          0.8636
0.62         0.9791       0.8776             0.8431          0.8600
0.63         0.9798       0.8899             0

### Building the test data for implementation of the model

In [180]:
# Load Kaggle test data
test_df = pd.read_csv(TEST_PATH)

# Parse time
test_df["time"]        = pd.to_datetime(test_df["time"], errors="coerce")
test_df["hour"]        = test_df["time"].dt.hour
test_df["day_of_week"] = test_df["time"].dt.dayofweek
test_df["is_weekend"]  = test_df["day_of_week"].isin([5, 6]).astype(int)

# Datetime cyclical features
X_dt_test    = add_cyclical_features(test_df)

# Regex features
X_regex_test = extract_regex_features(test_df["text"])

# Preprocess text + TF-IDF
test_clean   = preprocess(test_df["text"])
X_tfidf_test = vectoriser.transform(test_clean)

# ── LR test matrix — 14 dense cols, uses scaler ──────────────────────
X_dense_test_lr = scaler.transform(
    np.hstack([X_regex_test.values, X_dt_test.values])  # 14 cols
)
X_test_all = hstack([X_tfidf_test, X_dense_test_lr])

# ── XGBoost test matrix — 17 dense cols, uses scaler_xgb ─────────────
# Drop user cols if already merged from a previous run
for col in ["user_toxic_rate", "user_comment_count"]:
    if col in test_df.columns:
        test_df = test_df.drop(columns=[col])

test_df = test_df.merge(user_stats, on="username", how="left")
test_df["user_toxic_rate"]    = test_df["user_toxic_rate"].fillna(global_toxic_rate)
test_df["user_comment_count"] = test_df["user_comment_count"].fillna(0)

X_xgb_test = scaler_xgb.transform(np.hstack([
    X_regex_test.values,          # 9 cols
    X_dt_test.values,             # 5 cols
    test_df[user_cols].values     # 3 cols
]))                               # 17 cols total

print(f"Train shape:        {X_train_all.shape}")
print(f"LR test shape:      {X_test_all.shape}")    # must match train
print(f"XGBoost test shape: {X_xgb_test.shape}")    # (55711, 17)

Train shape:        (115125, 61311)
LR test shape:      (55711, 61311)
XGBoost test shape: (55711, 17)


In [181]:
best_lr_w, best_xgb_w, best_t, _ = best

y_test_proba_lr  = lr_all.predict_proba(X_test_all)[:, 1]
y_test_proba_xgb = xgb_model.predict_proba(X_xgb_test)[:, 1]

y_test_proba_final = best_lr_w * y_test_proba_lr + best_xgb_w * y_test_proba_xgb
y_test_pred_final  = (y_test_proba_final >= best_t).astype(int)

submission = pd.DataFrame({
    "Id":       test_df["Id"],
    "IS_TOXIC": y_test_pred_final
})
submission.to_csv("../outputs/submission_ensemble_v3.csv", index=False)
print("Saved: submission_ensemble_v3.csv ✅")
print(f"Predicted toxic:     {y_test_pred_final.sum():,} ({y_test_pred_final.mean():.2%})")
print(f"Predicted non-toxic: {(y_test_pred_final==0).sum():,}")
print(submission.head(10))

Saved: submission_ensemble_v3.csv ✅
Predicted toxic:     3,924 (7.04%)
Predicted non-toxic: 51,787
       Id  IS_TOXIC
0  169387         1
1  169388         0
2  169389         0
3  169390         0
4  169391         0
5  169392         0
6  169395         0
7  169404         0
8  169405         0
9  169406         0
